# Experimental BCC force constants overlaid on MOGA t-SNE

This notebook combines the top-five MOGA-Phonons BCC solutions from four dataframes with experimental Born-von Kármán force constants for BCC Cr, V, Fe, Ca, Ti, and Zr.

The first-neighbor variables are transformed as

\[
u = \alpha_1 + 2\beta_1,
\]

and

\[
v = \alpha_1 - \beta_1.
\]

The t-SNE embedding is fit to the combined set of MOGA and experimental points because standard t-SNE does not provide an exact out-of-sample transform. Publication-quality PDF and PNG figures are written to disk.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

%config InlineBackend.figure_format = "retina"

# -----------------------------------------------------------------------------
# Paths
# -----------------------------------------------------------------------------
DATA_DIR = Path("./dataframes")
EXPERIMENTAL_CSV = Path("./bcc_experimental_force_constants_alpha_beta.csv")

DATAFRAME_FILES = {
    "dataframe000013": DATA_DIR / "dataframe000013.pkl",
    "dataframe000014": DATA_DIR / "dataframe000014.pkl",
    "dataframe000015": DATA_DIR / "dataframe000015.pkl",
    "dataframe000016": DATA_DIR / "dataframe000016.pkl",
}

OUTPUT_DIR = Path("experimental_bcc_force_constants_moga_tsne")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------------------------
# Plot style
# -----------------------------------------------------------------------------
plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 600,
    "font.size": 18,
    "axes.labelsize": 22,
    "axes.titlesize": 24,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "legend.fontsize": 15,
    "axes.linewidth": 1.5,
    "xtick.major.size": 6,
    "ytick.major.size": 6,
    "xtick.direction": "in",
    "ytick.direction": "in",
})

RANDOM_STATE = 42


## Helper functions

In [ ]:
def find_first_existing(candidates, columns):
    for col in candidates:
        if col in columns:
            return col
    return None


def savefig(fig, stem):
    pdf = OUTPUT_DIR / f"{stem}.pdf"
    png = OUTPUT_DIR / f"{stem}.png"
    fig.savefig(pdf, bbox_inches="tight")
    fig.savefig(png, bbox_inches="tight")
    print(f"Saved: {pdf}")
    print(f"Saved: {png}")


def add_derived_columns(df):
    df = df.copy()
    df["u_alpha1_2beta1"] = df["alpha1"] + 2.0 * df["beta1"]
    df["v_alpha1_minus_beta1"] = df["alpha1"] - df["beta1"]
    denom = np.abs(df["alpha2"]) + np.abs(df["beta2"])
    df["r_alpha2"] = np.where(denom > 0, np.abs(df["alpha2"]) / denom, np.nan)
    return df


def infer_alpha0_from_sum_rule(alpha1, beta1, alpha2, beta2):
    # Approximate on-site term using the same 2NN sum-rule combination used in the GA.
    return -(8.0 * alpha1 + 2.0 * alpha2 + 4.0 * beta2)


# Unit conversions.
N_PER_M_TO_EV_PER_A2 = 0.06241509074460765
DYN_PER_CM_TO_EV_PER_A2 = 1.0e-3 * N_PER_M_TO_EV_PER_A2


## Load and filter MOGA solutions

The notebook selects the top five overall solutions ranked by `fitness_norm` for each `(dataset, mass, a)` group.

In [ ]:
frames = []
for label, path in DATAFRAME_FILES.items():
    if not path.exists():
        raise FileNotFoundError(f"Could not find {path}. Update DATA_DIR or DATAFRAME_FILES.")
    tmp = pd.read_pickle(path).copy()
    tmp["dataset"] = label
    print(f"{label}: {tmp.shape[0]:,} rows, {tmp.shape[1]:,} columns")
    frames.append(tmp)

df_all = pd.concat(frames, ignore_index=True)
print(f"\nAggregated dataframe before filtering: {df_all.shape[0]:,} rows")

mass_col = find_first_existing(["mass", "m"], df_all.columns)
alat_col = find_first_existing(["a_val", "alat", "a_latt", "lattice_parameter"], df_all.columns)
rank_col = find_first_existing(["fitness_norm", "normalized_fitness", "fitness"], df_all.columns)

required = [mass_col, alat_col, rank_col, "alpha0", "alpha1", "beta1", "alpha2", "beta2"]
missing = [c for c in required if c is None or c not in df_all.columns]
if missing:
    raise KeyError(f"Missing required columns: {missing}")

if mass_col != "mass":
    df_all = df_all.rename(columns={mass_col: "mass"})
if alat_col != "a_val":
    df_all = df_all.rename(columns={alat_col: "a_val"})

sort_ascending = False
moga_top = (
    df_all.sort_values(rank_col, ascending=sort_ascending)
          .groupby(["dataset", "mass", "a_val"], as_index=False, group_keys=False)
          .head(5)
          .reset_index(drop=True)
)

moga_top = add_derived_columns(moga_top)
moga_top["source"] = "MOGA"
moga_top["element"] = "MOGA"
moga_top["temperature_K"] = np.nan

print(f"Top-5 overall selected dataframe: {moga_top.shape[0]:,} rows")
moga_top[["dataset", "mass", "a_val", "alpha0", "alpha1", "beta1", "alpha2", "beta2", "u_alpha1_2beta1", "v_alpha1_minus_beta1"]].head()


## Build experimental force-constant table

The CSV contains the previously compiled Cr, V, and Fe data in eV/Å$^2$. Ca is entered from the table in N/m. Ti and Zr are entered from the tables in dyn/cm.

The sign convention used here is the same one used in the previous comparison: reported off-site BvK constants are multiplied by $-1$ to match the MOGA convention.


In [ ]:
experimental_rows = []

# Existing Cr, V, Fe CSV in eV/A^2.
if EXPERIMENTAL_CSV.exists():
    exp_csv = pd.read_csv(EXPERIMENTAL_CSV)
else:
    # Compatibility fallback for the filename created in this ChatGPT session.
    fallback = Path("./bcc_experimental_force_constants_alpha_beta(1).csv")
    if fallback.exists():
        exp_csv = pd.read_csv(fallback)
    else:
        raise FileNotFoundError("Could not find the experimental Cr/V/Fe CSV. Update EXPERIMENTAL_CSV.")

# Normalize common column names.
rename_map = {
    "alpha1_eV_A2": "alpha1",
    "beta1_eV_A2": "beta1",
    "alpha2_eV_A2": "alpha2",
    "beta2_eV_A2": "beta2",
}
exp_csv = exp_csv.rename(columns=rename_map)
for _, row in exp_csv.iterrows():
    alpha1, beta1, alpha2, beta2 = row["alpha1"], row["beta1"], row["alpha2"], row["beta2"]
    experimental_rows.append({
        "element": row["element"],
        "temperature_K": float(row["temperature_K"]),
        "alpha1": alpha1,
        "beta1": beta1,
        "alpha2": alpha2,
        "beta2": beta2,
        "alpha0": infer_alpha0_from_sum_rule(alpha1, beta1, alpha2, beta2),
        "source": "experiment",
    })

# Ca data from table, N/m. Values are off-site BvK constants in the original sign convention.
ca_data_N_m = [
    {"element": "Ca", "temperature_K": 726.0, "one_xx": 4.014, "one_xy": 3.503, "two_xx": 1.443, "two_xy": -1.182},
    {"element": "Ca", "temperature_K": 750.0, "one_xx": 3.936, "one_xy": 3.582, "two_xx": 1.441, "two_xy": -0.926},
]
for d in ca_data_N_m:
    alpha1 = -d["one_xx"] * N_PER_M_TO_EV_PER_A2
    beta1  = -d["one_xy"] * N_PER_M_TO_EV_PER_A2
    alpha2 = -d["two_xx"] * N_PER_M_TO_EV_PER_A2
    beta2  = -d["two_xy"] * N_PER_M_TO_EV_PER_A2
    experimental_rows.append({
        "element": d["element"],
        "temperature_K": d["temperature_K"],
        "alpha1": alpha1,
        "beta1": beta1,
        "alpha2": alpha2,
        "beta2": beta2,
        "alpha0": infer_alpha0_from_sum_rule(alpha1, beta1, alpha2, beta2),
        "source": "experiment",
    })

# Ti beta phase at 1020 C = 1293.15 K, dyn/cm.
ti_data_dyn_cm = [
    {"element": "Ti", "temperature_K": 1020.0 + 273.15, "one_xx": 8354.7, "one_xy": 7774.9, "two_xx": 4846.7, "two_yy": -2355.9},
]
for d in ti_data_dyn_cm:
    alpha1 = -d["one_xx"] * DYN_PER_CM_TO_EV_PER_A2
    beta1  = -d["one_xy"] * DYN_PER_CM_TO_EV_PER_A2
    alpha2 = -d["two_xx"] * DYN_PER_CM_TO_EV_PER_A2
    beta2  = -d["two_yy"] * DYN_PER_CM_TO_EV_PER_A2
    experimental_rows.append({
        "element": d["element"],
        "temperature_K": d["temperature_K"],
        "alpha1": alpha1,
        "beta1": beta1,
        "alpha2": alpha2,
        "beta2": beta2,
        "alpha0": infer_alpha0_from_sum_rule(alpha1, beta1, alpha2, beta2),
        "source": "experiment",
    })

# Zr beta phase at various temperatures, dyn/cm.
# The table labels the second-neighbor rows as 2_yy twice; the first is interpreted as 2_xx.
zr_data_dyn_cm = [
    {"element": "Zr", "temperature_K":  915.0 + 273.15, "one_xx": 7351.7, "one_xy": 8565.5, "two_xx": 4967.6, "two_yy": -1952.8},
    {"element": "Zr", "temperature_K": 1210.0 + 273.15, "one_xx": 7798.4, "one_xy": 8341.7, "two_xx": 4960.5, "two_yy": -2170.2},
    {"element": "Zr", "temperature_K": 1610.0 + 273.15, "one_xx": 8140.2, "one_xy": 7993.8, "two_xx": 4564.7, "two_yy": -1898.2},
]
for d in zr_data_dyn_cm:
    alpha1 = -d["one_xx"] * DYN_PER_CM_TO_EV_PER_A2
    beta1  = -d["one_xy"] * DYN_PER_CM_TO_EV_PER_A2
    alpha2 = -d["two_xx"] * DYN_PER_CM_TO_EV_PER_A2
    beta2  = -d["two_yy"] * DYN_PER_CM_TO_EV_PER_A2
    experimental_rows.append({
        "element": d["element"],
        "temperature_K": d["temperature_K"],
        "alpha1": alpha1,
        "beta1": beta1,
        "alpha2": alpha2,
        "beta2": beta2,
        "alpha0": infer_alpha0_from_sum_rule(alpha1, beta1, alpha2, beta2),
        "source": "experiment",
    })

exp_df = pd.DataFrame(experimental_rows)
exp_df = add_derived_columns(exp_df)
exp_df["dataset"] = "experiment"
exp_df["mass"] = np.nan
exp_df["a_val"] = np.nan

# Save the compiled experimental table.
exp_table_path = OUTPUT_DIR / "compiled_experimental_bcc_force_constants_with_ti_zr_ca.csv"
exp_df.to_csv(exp_table_path, index=False)
print(f"Saved compiled experimental table: {exp_table_path}")

exp_df[["element", "temperature_K", "alpha1", "beta1", "u_alpha1_2beta1", "v_alpha1_minus_beta1", "alpha2", "beta2", "alpha0"]]


## Prepare combined feature matrix

In [ ]:
FEATURES = ["alpha0", "u_alpha1_2beta1", "v_alpha1_minus_beta1", "alpha2", "beta2"]

moga_features = moga_top[FEATURES + ["source", "element", "dataset", "mass", "a_val", "temperature_K", "r_alpha2"]].copy()
exp_features = exp_df[FEATURES + ["source", "element", "dataset", "mass", "a_val", "temperature_K", "r_alpha2"]].copy()
combined = pd.concat([moga_features, exp_features], ignore_index=True)
combined = combined.replace([np.inf, -np.inf], np.nan).dropna(subset=FEATURES).reset_index(drop=True)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(combined[FEATURES].to_numpy())

is_moga = combined["source"].eq("MOGA").to_numpy()
is_exp = ~is_moga

print("Feature columns:", FEATURES)
print(f"Combined rows: {combined.shape[0]:,}")
print(f"MOGA rows: {is_moga.sum():,}")
print(f"Experimental rows: {is_exp.sum():,}")
combined.tail(10)


## PCA reference projection

This is not the main t-SNE result, but it is useful because PCA provides interpretable axes.

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
pc = pca.fit_transform(X_scaled)
combined["PC1"] = pc[:, 0]
combined["PC2"] = pc[:, 1]
print("Explained variance ratio:", pca.explained_variance_ratio_)


In [ ]:
element_markers = {
    "Ca": "D", "Cr": "s", "Fe": "o", "V": "^", "Ti": "P", "Zr": "X",
}
element_colors = {
    "Ca": "tab:blue", "Cr": "tab:purple", "Fe": "tab:red", "V": "tab:green", "Ti": "tab:orange", "Zr": "tab:brown",
}

fig, ax = plt.subplots(figsize=(7.2, 6.0))
ax.scatter(combined.loc[is_moga, "PC1"], combined.loc[is_moga, "PC2"],
           s=28, c="0.75", alpha=0.35, edgecolors="none", label="MOGA top-5 solutions")

for element in ["Ca", "Cr", "Fe", "V", "Ti", "Zr"]:
    mask = is_exp & combined["element"].eq(element).to_numpy()
    if not mask.any():
        continue
    ax.scatter(combined.loc[mask, "PC1"], combined.loc[mask, "PC2"],
               s=130, marker=element_markers[element], c=element_colors[element],
               edgecolors="black", linewidths=1.2, label=element, zorder=5)

ax.axhline(0, color="0.8", lw=1.2, zorder=0)
ax.axvline(0, color="0.8", lw=1.2, zorder=0)
ax.set_xlabel(f"PC1 ({100*pca.explained_variance_ratio_[0]:.1f}%)")
ax.set_ylabel(f"PC2 ({100*pca.explained_variance_ratio_[1]:.1f}%)")
ax.set_title("Experimental BCC force constants projected onto MOGA PCA")
ax.legend(frameon=False, loc="best")
ax.tick_params(top=True, right=True)
fig.tight_layout()
savefig(fig, "pca_experimental_overlay_ca_cr_fe_v_ti_zr")
plt.show()


## t-SNE embeddings

The main manuscript figure uses perplexity 20. Additional perplexities are generated to check robustness.

In [ ]:
PERPLEXITIES = [10, 20, 30]
embeddings = {}

for perplexity in PERPLEXITIES:
    tsne = TSNE(
        n_components=2,
        perplexity=perplexity,
        learning_rate="auto",
        init="pca",
        random_state=RANDOM_STATE,
        metric="euclidean",
    )
    emb = tsne.fit_transform(X_scaled)
    embeddings[perplexity] = emb
    combined[f"tsne1_p{perplexity}"] = emb[:, 0]
    combined[f"tsne2_p{perplexity}"] = emb[:, 1]
    print(f"Finished t-SNE with perplexity={perplexity}")


In [ ]:
def plot_tsne_experimental_overlay(perplexity=20):
    xcol = f"tsne1_p{perplexity}"
    ycol = f"tsne2_p{perplexity}"

    fig, ax = plt.subplots(figsize=(7.2, 6.0))
    ax.scatter(combined.loc[is_moga, xcol], combined.loc[is_moga, ycol],
               s=28, c="0.75", alpha=0.35, edgecolors="none", label="MOGA top-5 solutions")

    for element in ["Ca", "Cr", "Fe", "V", "Ti", "Zr"]:
        mask = is_exp & combined["element"].eq(element).to_numpy()
        if not mask.any():
            continue
        ax.scatter(combined.loc[mask, xcol], combined.loc[mask, ycol],
                   s=135, marker=element_markers[element], c=element_colors[element],
                   edgecolors="black", linewidths=1.2, label=element, zorder=5)

    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")
    ax.set_title(fr"Experimental BCC force constants on MOGA t-SNE, $p={perplexity}$")
    ax.legend(frameon=False, loc="best")
    ax.tick_params(top=True, right=True)
    fig.tight_layout()
    savefig(fig, f"tsne_p{perplexity}_experimental_overlay_ca_cr_fe_v_ti_zr")
    plt.show()

for p in PERPLEXITIES:
    plot_tsne_experimental_overlay(p)


In [ ]:
def plot_tsne_temperature(perplexity=20):
    xcol = f"tsne1_p{perplexity}"
    ycol = f"tsne2_p{perplexity}"

    fig, ax = plt.subplots(figsize=(7.6, 6.0))
    ax.scatter(combined.loc[is_moga, xcol], combined.loc[is_moga, ycol],
               s=28, c="0.75", alpha=0.25, edgecolors="none", label="MOGA top-5 solutions")

    sc = ax.scatter(combined.loc[is_exp, xcol], combined.loc[is_exp, ycol],
                    c=combined.loc[is_exp, "temperature_K"], s=125,
                    cmap="viridis", edgecolors="black", linewidths=1.0, zorder=5)

    # Label non-Fe elements and sparse Fe endpoints to reduce clutter.
    for _, row in combined.loc[is_exp].iterrows():
        element = row["element"]
        T = row["temperature_K"]
        if element != "Fe" or T in [combined.loc[(is_exp) & combined["element"].eq("Fe"), "temperature_K"].min(),
                             combined.loc[(is_exp) & combined["element"].eq("Fe"), "temperature_K"].max()]:
            ax.text(row[xcol] + 0.8, row[ycol] + 0.8, f"{element} {T:.0f} K", fontsize=10)

    cbar = fig.colorbar(sc, ax=ax)
    cbar.set_label(r"$T$ (K)")
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")
    ax.set_title(fr"Experimental BCC force constants colored by temperature, $p={perplexity}$")
    ax.tick_params(top=True, right=True)
    fig.tight_layout()
    savefig(fig, f"tsne_p{perplexity}_experimental_overlay_colored_by_temperature")
    plt.show()

plot_tsne_temperature(20)


## Experimental force-constant maps

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 6.0))
ax.scatter(moga_top["alpha2"], moga_top["beta2"], s=24, c="0.75", alpha=0.30, edgecolors="none", label="MOGA top-5 solutions")
for element in ["Ca", "Cr", "Fe", "V", "Ti", "Zr"]:
    sub = exp_df[exp_df["element"] == element]
    if sub.empty:
        continue
    ax.scatter(sub["alpha2"], sub["beta2"], s=135, marker=element_markers[element],
               c=element_colors[element], edgecolors="black", linewidths=1.2, label=element, zorder=5)
ax.axhline(0, color="0.8", lw=1.2, zorder=0)
ax.axvline(0, color="0.8", lw=1.2, zorder=0)
ax.set_xlabel(r"$\alpha_2$ (eV/$\AA^2$)")
ax.set_ylabel(r"$\beta_2$ (eV/$\AA^2$)")
ax.set_title("Experimental second-neighbor force constants")
ax.legend(frameon=False, loc="best")
ax.tick_params(top=True, right=True)
fig.tight_layout()
savefig(fig, "experimental_second_neighbor_force_constants_ca_cr_fe_v_ti_zr")
plt.show()

fig, ax = plt.subplots(figsize=(7.2, 6.0))
ax.scatter(moga_top["u_alpha1_2beta1"], moga_top["v_alpha1_minus_beta1"], s=24, c="0.75", alpha=0.30, edgecolors="none", label="MOGA top-5 solutions")
for element in ["Ca", "Cr", "Fe", "V", "Ti", "Zr"]:
    sub = exp_df[exp_df["element"] == element]
    if sub.empty:
        continue
    ax.scatter(sub["u_alpha1_2beta1"], sub["v_alpha1_minus_beta1"], s=135, marker=element_markers[element],
               c=element_colors[element], edgecolors="black", linewidths=1.2, label=element, zorder=5)
ax.axhline(0, color="0.8", lw=1.2, zorder=0)
ax.axvline(0, color="0.8", lw=1.2, zorder=0)
ax.set_xlabel(r"$\alpha_1 + 2\beta_1$ (eV/$\AA^2$)")
ax.set_ylabel(r"$\alpha_1 - \beta_1$ (eV/$\AA^2$)")
ax.set_title("Experimental first-neighbor transformed coordinates")
ax.legend(frameon=False, loc="best")
ax.tick_params(top=True, right=True)
fig.tight_layout()
savefig(fig, "experimental_first_neighbor_transformed_coordinates_ca_cr_fe_v_ti_zr")
plt.show()


## Notes

- The Ti and Zr tables report force constants in dyn/cm. The conversion used is

\[
1\;\mathrm{dyn/cm}=10^{-3}\;\mathrm{N/m}=6.241509\times 10^{-5}\;\mathrm{eV}/\AA^2.
\]

- The Ca table reports force constants in N/m. The conversion used is

\[
1\;\mathrm{N/m}=0.06241509\;\mathrm{eV}/\AA^2.
\]

- The off-site experimental constants are multiplied by \(-1\) to match the sign convention used in the MOGA dataframes.

- The experimental on-site parameter \(\alpha_0\) is inferred from the two-neighbor acoustic sum-rule combination used in the GA fitness,

\[
\alpha_0 \approx -\left(8\alpha_1 + 2\alpha_2 + 4\beta_2\right).
\]

- The Zr table appears to list the second-neighbor row label as \(2_{yy}\) twice; the first of these is interpreted as \(2_{xx}\), consistent with the BCC second-neighbor shell.
